# Release notes
Paths and language were normalized for this source release. Historical training protocols remain documented in `REVIEW-1023.md`; retrain after any protocol corrections.


In [ ]:
from pathlib import Path
import sys
import copy
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "models/DeepLearnModel.py").is_file() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the MMHRP-CAMoPE-1023 project.")
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install rdkit==2024.9.5
# !pip install torch_geometric==2.5.3

In [ ]:
import os, sys

# Inspect the current working directory
print("Current working directory:", os.getcwd())

# Inspect the Python module search path
print("Python module search path:")
for p in sys.path:
    print(p)


In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader
doc_name = str(PROJECT_ROOT)
sys.path.append(doc_name)
from utils.rxn import *
from utils.molecule import *
from utils.dataset_analysis import *
from models.DeepLearnModel import *
import time
from tqdm import tqdm
import datetime
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# rs_list = [1,2,3,4,5]
rs_list = [1]
eval_metrics = np.zeros((1+len(rs_list), 6))
columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
index = []
for rs in rs_list:
  index.append("%s" % rs)
index.append("avg±std")
eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

In [ ]:
for rs in rs_list:
    # 1. import data
    data = pd.read_excel("%s/data/Heck/Heck_fp_clean.xlsx" % doc_name)
    random_state = rs
    data = data.sample(random_state=random_state, frac=1).reset_index(drop=True)

    # Generate one-hot features
    df_temp_onehot = pd.get_dummies(data["temperature /C"], prefix="Temp")
    df_time_onehot = pd.get_dummies(data["time /h"], prefix="Time")
    
    # Concatenate the features
    df_condition_onehot = pd.concat([df_temp_onehot, df_time_onehot], axis=1)
    CondDim = df_condition_onehot.shape[1]
    
    # Convert to a tensor
    condition_tensor_all = torch.tensor(df_condition_onehot.values, dtype=torch.float32)
                                    
    # 2. build dataset & dataloader
    rxn_dataset = list()
    rxn_list = df_to_rxn_list(data)
    len_drfp = 0

    for i in tqdm(range(data.shape[0])):
        rxn = rxn_list[i]
    
        # DRFP
        drfp = torch.tensor(read_drfp(data.loc[i]["drfp"]), dtype=torch.float32)
        len_drfp = drfp.shape[0]
    
        # One-hot encoded temperature and time
        cond_vec = condition_tensor_all[i]    # shape = (CondDim,)
    
        # Concatenate into one input vector
        features = torch.cat([drfp, cond_vec], dim=0)
    
        # label
        y = rxn.rxn_yield / 100
    
        rxn_dataset.append([features, y])
        
    # report
    base_dir = r"C:\Users\29868\Desktop\python\Found_tri-modal\autodl\HeckLit-Code-main\exp\Heck_Benchmark"
    dir_path = os.path.join(base_dir, f"Heck_ANN_rs=1b")
    
    os.makedirs(dir_path, exist_ok=True)
    report_path = os.path.join(dir_path, "Model_Training_Report.txt")
    f = open(report_path, mode="w")
    # split of train & test set
    ratio = 0.8
    batch_size = 1500
    batch = len(rxn_dataset)
    train_set = rxn_dataset[0: int(ratio * batch)]
    test_set = rxn_dataset[int(ratio * batch) + 1:]

    # data_loader
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=True)

    train_R2 = list()
    train_RMSE = list()
    train_MAE = list()
    test_R2 = list()
    test_RMSE = list()
    test_MAE = list()
    pred = list()
    true = list()

    # 3. training of the model
    # params
    t = 2000
    lr = 1e-4

    # model
    model = ANN(input_size=len_drfp+ CondDim)
    opti = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    # use gpu
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device("cpu")
    model = model.to(device)
    criterion = nn.MSELoss()

    # writedown params
    f.write("params:\n")
    f.write("random_state=%s\n" % random_state)
    f.write("ratio=%s\n" % ratio)
    f.write("batch_size=%s\n" % batch_size)
    f.write("t=%s\n" % t)
    f.write("lr=%s\n" % lr)

    # Training
    # best performance
    best = [0, 0, 0, 0, 0, 0, [], []]  # train_R2, train_RMSE, train_MAE, test_R2, test_RMSE, test_MAE, test_predict, test_true

    f.write("\nStart training\n")

    for epoch in tqdm(range(t)):
        # Training
        global_loss = torch.tensor([0.])

        for data in train_loader:
            x = data[:-1][0].to(device)
            y = torch.unsqueeze(data[-1], dim=1).to(device)
            loss = criterion(model.forward(x).float(), y.float())
            opti.zero_grad()
            loss.backward()
            opti.step()
            global_loss += loss.item()

        # record of loss during training
        # performance in train set
        with torch.no_grad():
            pred = list()
            true = list()
            for data in train_loader:
                x = data[:-1][0].to(device)
                tr = torch.unsqueeze(data[-1], dim=1).to(device)
                pr = list(model.forward(x).cpu().detach().numpy())
                pred += pr
                true += list(tr.cpu().detach().numpy())
            train_R2.append(R2(np.array(pred), np.array(true)))
            train_RMSE.append(RMSE(np.array(pred), np.array(true)))
            train_MAE.append(MAE(np.array(pred), np.array(true)))

        # performance in test set
        with torch.no_grad():
            pred = list()
            true = list()
            for data in test_loader:
                x = data[:-1][0].to(device)
                tr = torch.unsqueeze(data[-1], dim=1).to(device)
                pr = list(model.forward(x).cpu().detach().numpy())
                pred += pr
                true += list(tr.cpu().detach().numpy())
            test_R2.append(R2(np.array(pred), np.array(true)))
            test_RMSE.append(RMSE(np.array(pred), np.array(true)))
            test_MAE.append(MAE(np.array(pred), np.array(true)))

            if epoch == 0 or test_R2[-1] >= best[3]:
                best = [train_R2[-1], train_RMSE[-1], train_MAE[-1], test_R2[-1], test_RMSE[-1], test_MAE[-1], pred, true]

        # write report
        f.write("Epoch:%d loss: %f, R2:train set %.3f\ttest set %.3f\n" % (
        epoch + 1, global_loss / batch_size, train_R2[-1], test_R2[-1]))

    # 4.Evaluation
    f.write("\n")
    # Performance in train set
    f.write("R2 of train set is:%.3f+-%f\tbest:%f\n" % (
    np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
    f.write("RMSE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))
    f.write("MAE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_MAE[-10:]).mean(), np.array(train_MAE[-10:]).std(), best[2]))

    # Performance in test set
    f.write("R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (
    np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
    f.write("RMSE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))
    f.write("MAE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_MAE[-10:]).mean(), np.array(test_MAE[-10:]).std(), best[5]))

    f.close()

    # Performance in train set
    print("R2 of train set is:%.3f+-%f\tbest:%f\n" % (
    np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
    print("RMSE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))

    # Performance in test set
    print("R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (
    np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
    print("RMSE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))

    eval_metrics.loc["%s" % rs]["train_R2"] = best[0]
    eval_metrics.loc["%s" % rs]["train_RMSE"] = best[1]
    eval_metrics.loc["%s" % rs]["train_MAE"] = best[2]
    eval_metrics.loc["%s" % rs]["test_R2"] = best[3]
    eval_metrics.loc["%s" % rs]["test_RMSE"] = best[4]
    eval_metrics.loc["%s" % rs]["test_MAE"] = best[5]

    # 5.Figure
    import matplotlib.pyplot as plt
    fig = plt.figure(dpi=500, figsize=(20, 7))

    # Training Fig
    plt.subplot(1, 2, 1)
    steps = np.linspace(1, t, t)
    plt.plot(steps, train_R2, color=[236 / 255, 164 / 255, 124 / 255])
    plt.plot(steps, test_R2, color=[117 / 255, 157 / 255, 219 / 255])
    # Beautify
    plt.legend(["train set R$^2$", "test set R$^2$"], loc="upper left", prop={'size': 10})
    plt.xlabel("Epoch", fontsize=10)
    plt.ylabel("R$^2$ value", fontsize=10)
    plt.title("The R$^2$ value during training", fontsize=13)

    # Test set performance
    plt.subplot(1, 2, 2)
    tr = np.array(best[-1]).flatten() * 100
    pr = np.array(best[-2]).flatten() * 100
    plt.scatter(pr, tr, alpha=0.7, marker=".")
    plt.xlabel("Predicted Yield", fontsize=10)
    plt.ylabel("Observed Yield", fontsize=10)
    x = np.linspace(0, 100, 100)
    y = np.linspace(0, 100, 100)
    plt.plot(x, y, linestyle="--", color="r")
    plt.title("Test set performance", fontsize=15)

    fig.suptitle("DRFP for Heck dataset", fontsize=16)
    plt.tight_layout()
    plt.savefig("%s/Performance_Figure.png" % dir_path)
    plt.show()

In [ ]:
# Evaluation metrics report
for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
  for j in range(eval_metrics.shape[1]):
    eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
eval_metrics.to_csv(r"C:\Users\29868\Desktop\python\Found_tri-modal\autodl\HeckLit-Code-main\exp\Heck_Benchmark\Heck_ANN_rs=1b\Heck_ANN_rs=1b.csv")
print(eval_metrics)